# Fine-tune SmolLM2-135M on Teaching Q&A Dataset
Run this notebook on a Colab GPU instance (T4 is enough).

In [ ]:
!pip install -q transformers datasets trl accelerate sentencepiece

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer, SFTConfig

BASE_MODEL  = "HuggingFaceTB/SmolLM2-135M-Instruct"
DATA_PATH   = "/content/training_data_clean.jsonl"
OUTPUT_DIR  = "/content/output"
MAX_SEQ_LEN = 512

In [ ]:
# Load dataset
records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} records")

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.model_max_length = MAX_SEQ_LEN
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

In [ ]:
# Format records into chat template messages
def format_record(record):
    messages = [
        {"role": "user",      "content": record["question"]},
        {"role": "assistant", "content": record["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list([format_record(r) for r in records])
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(dataset)

In [ ]:
# Train
config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    warmup_steps=50,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

In [ ]:
# Save model and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# Convert to GGUF and quantize
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
!pip install -q gguf

# Convert to F16 GGUF
!python /tmp/llama.cpp/convert_hf_to_gguf.py {OUTPUT_DIR} --outfile smollm2-tutor-f16.gguf --outtype f16

# Build llama-quantize binary
!cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_NATIVE=OFF -DLLAMA_BUILD_TESTS=OFF 2>/dev/null
!cmake --build /tmp/llama.cpp/build --target llama-quantize -j$(nproc) 2>/dev/null

# Quantize to Q4_K_M
!/tmp/llama.cpp/build/bin/llama-quantize smollm2-tutor-f16.gguf smollm2-tutor-q4.gguf Q4_K_M

print("Done — download smollm2-tutor-q4.gguf")

In [ ]:
# Convert to GGUF and quantize
# Clone llama.cpp in Colab to use its conversion script
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
!pip install -q gguf

!python /tmp/llama.cpp/convert_hf_to_gguf.py {OUTPUT_DIR} --outfile smollm2-tutor-f16.gguf --outtype f16
!python /tmp/llama.cpp/llama-quantize smollm2-tutor-f16.gguf smollm2-tutor-q4.gguf Q4_K_M

print("Done — download smollm2-tutor-q4.gguf")